# Create Unity Catalog Layer

Creates or repairs Unity Catalog metadata for one selected Ampere layer from the canonical table contract.

Steps performed:
1. Select layer and contract path.
2. Load `.env`, UC runtime config, and canonical table contract through `tools/py_utils`.
3. Ensure the catalog and required schemas exist.
4. Replace external table metadata for the selected layer.
5. Write UC input and execution report files under `tools/uc/reports/`.


In [ ]:
# -----------------------------------------------------------------------------
# Manual setup
# -----------------------------------------------------------------------------
LAYER = "gold"  # bronze | silver | gold
CONTRACT_PATH = "tools/uc/contracts/ampere_tables.json"
REPORT_DIR = "tools/uc/reports"
VERBOSE = True


In [ ]:
# -----------------------------------------------------------------------------
# Imports and project path setup
# -----------------------------------------------------------------------------
from pathlib import Path
import json
import sys

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "pyproject.toml").exists():
        sys.path.insert(0, str(candidate))
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError("Could not find project root from current notebook path.")

from tools.py_utils.paths import project_root  # noqa: E402
from tools.py_utils.uc_client import UCClient  # noqa: E402
from tools.py_utils.uc_config import load_uc_bulk_config, uc_client_config  # noqa: E402
from tools.py_utils.uc_contracts import layer_schemas, layer_uc_payload, layers, load_contract  # noqa: E402
from tools.py_utils.uc_registration import (  # noqa: E402
    empty_registration_report,
    ensure_catalog,
    ensure_schemas,
    load_table_contracts,
    replace_external_tables,
    write_report,
)


In [ ]:
# -----------------------------------------------------------------------------
# Prepare selected UC layer
# -----------------------------------------------------------------------------
root = project_root(__file__ if "__file__" in globals() else PROJECT_ROOT)
contract = load_contract(CONTRACT_PATH)
contract_layers = layers(contract)
if LAYER not in contract_layers:
    raise ValueError(f"Unknown layer {LAYER!r}. Available: {sorted(contract_layers)}")

config = load_uc_bulk_config()
uc = UCClient(uc_client_config(config, verbose=VERBOSE))
report = empty_registration_report(
    catalog=uc.config.catalog,
    layer_name=LAYER,
    report_format=contract_layers[LAYER].get("format", "DELTA").upper(),
)

ensure_catalog(uc, comment=f"Ampere {uc.config.catalog} catalog", report=report)
ensure_schemas(uc, layer_schemas(contract, LAYER), report=report)

payload_path = root / REPORT_DIR / f"uc_{LAYER}_input.json"
payload_path.parent.mkdir(parents=True, exist_ok=True)
payload_path.write_text(
    json.dumps(layer_uc_payload(contract, LAYER), indent=2) + "\n",
    encoding="utf-8",
)
specs = load_table_contracts(payload_path)
replace_external_tables(uc, specs, report)

report_path = root / REPORT_DIR / f"uc_{LAYER}_report.json"
report["config_catalog"] = config["catalog"]
write_report(report, report_path)
report_path
